In [8]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [9]:
date = read_table("select * from sc_gold.dim_date")
ut = read_table("select * from sc_gold.dim_underemp_type")

In [10]:
df = read_table("select * from sc_silver.forecast_underemployment")
df

,date,underemp_type,underemp_rate_pct,source
0,2016-01-01,skill,22.931109,historical
1,2016-04-01,skill,23.339098,historical
2,2016-07-01,skill,23.735468,historical
3,2016-10-01,skill,24.120710,historical
4,2017-01-01,skill,24.495286,historical
...,...,...,...,...
115,2029-10-01,time,1.218491,forecast
116,2030-01-01,time,1.231630,forecast
117,2030-04-01,time,1.244541,forecast
118,2030-07-01,time,1.257229,forecast


In [11]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df=df.merge(
    ut[["underemp_type", "underemp_type_id"]],
    on="underemp_type",
    how="left"
)

df_final = df.drop(columns=["date","underemp_type"])
id_cols = ["date_id", "underemp_type_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [12]:
df_final["uer_id"] = ["UER" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["uer_id"] + [c for c in df_final.columns if c != "uer_id"]]
df_final

,uer_id,date_id,underemp_type_id,underemp_rate_pct,source
0,UER0001,DT001,UT001,22.931109,historical
1,UER0002,DT002,UT001,23.339098,historical
2,UER0003,DT003,UT001,23.735468,historical
3,UER0004,DT004,UT001,24.120710,historical
4,UER0005,DT005,UT001,24.495286,historical
...,...,...,...,...,...
115,UER0116,DT056,UT002,1.218491,forecast
116,UER0117,DT057,UT002,1.231630,forecast
117,UER0118,DT058,UT002,1.244541,forecast
118,UER0119,DT059,UT002,1.257229,forecast


In [13]:
write_table(df_final, "sc_gold", "fact_underemp_rate")

Table sc_gold.fact_underemp_rate written successfully.
